In [ ]:
# Cell 1 — Setup
from google.colab import drive
drive.mount('/content/drive')

import json
import numpy as np
from scipy import stats
from scipy.stats import mannwhitneyu
import os

DRIVE_BASE = "/content/drive/MyDrive/medrag"
OUTPUT_DIR = f"{DRIVE_BASE}/biomistral_eas_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load T2 and T4 results
with open(f"{DRIVE_BASE}/biomistral_trial2_outputs/trial2_full_results.json") as f:
    t2_data = json.load(f)

with open(f"{DRIVE_BASE}/biomistral_trial4_outputs/trial4_full_results.json") as f:
    t4_data = json.load(f)

n_layers  = t2_data["n_layers"]
n_samples = t2_data["n_samples"]

print(f"T2 loaded: {len(t2_data['samples'])} samples")
print(f"T4 loaded: {len(t4_data['samples'])} samples")
print(f"Layers: {n_layers}")

In [ ]:
# Cell 2 — Align samples and extract trajectories

# Align by pubid to be safe
t2_by_pubid = {r["pubid"]: r for r in t2_data["samples"]}
t4_by_pubid = {r["pubid"]: r for r in t4_data["samples"]}
common_pubids = sorted(set(t2_by_pubid) & set(t4_by_pubid))

print(f"Common samples (by pubid): {len(common_pubids)}")

labels    = []
t2_matrix = []   # (n_samples, n_layers)
t4_matrix = []   # (n_samples, n_layers)

for pubid in common_pubids:
    t2r = t2_by_pubid[pubid]
    t4r = t4_by_pubid[pubid]
    labels.append(t2r["label"])
    t2_matrix.append(t2r["prob_mass_trajectory"])
    t4_matrix.append(t4r["kl_trajectory"])

t2_matrix = np.array(t2_matrix)   # (n_samples, n_layers)
t4_matrix = np.array(t4_matrix)   # (n_samples, n_layers)
labels    = np.array(labels)

yes_mask = labels == "yes"
no_mask  = labels == "no"

print(f"yes: {yes_mask.sum()} | no: {no_mask.sum()}")
print(f"T2 matrix shape: {t2_matrix.shape}")
print(f"T4 matrix shape: {t4_matrix.shape}")

In [ ]:
# Cell 3 — Z-score and assemble EAS

# Z-score each signal across samples at each layer independently
def zscore_matrix(matrix):
    mean = matrix.mean(axis=0, keepdims=True)
    std  = matrix.std(axis=0, keepdims=True)
    std[std < 1e-8] = 1e-8  # prevent division by zero at flat layers
    return (matrix - mean) / std

z_t2 = zscore_matrix(t2_matrix)   # (n_samples, n_layers)
z_t4 = zscore_matrix(t4_matrix)   # (n_samples, n_layers)

# EAS composite: equal weight average of z-scored T2 and T4
# T1 excluded (no label discrimination in both models)
eas  = (z_t2 + z_t4) / 2.0        # (n_samples, n_layers)

print("EAS assembled.")
print(f"EAS shape: {eas.shape}")
print(f"EAS mean (yes): {eas[yes_mask].mean():.4f}")
print(f"EAS mean (no):  {eas[no_mask].mean():.4f}")

In [ ]:
# Cell 4 — Final-layer EAS statistics

final_eas_yes = eas[yes_mask, -1]
final_eas_no  = eas[no_mask,  -1]

print(f"Final-layer EAS (layer {n_layers-1})")
print(f"  yes: mean={final_eas_yes.mean():.4f}, std={final_eas_yes.std():.4f}, n={len(final_eas_yes)}")
print(f"  no:  mean={final_eas_no.mean():.4f},  std={final_eas_no.std():.4f},  n={len(final_eas_no)}")

# Mann-Whitney U test
mwu_stat, mwu_p = mannwhitneyu(final_eas_yes, final_eas_no, alternative="two-sided")
print(f"\nMann-Whitney U: stat={mwu_stat:.1f}, p={mwu_p:.4f}")
print(f"Significant (p<0.05): {mwu_p < 0.05}")

# Welch's t-test
t_stat, t_p = stats.ttest_ind(final_eas_yes, final_eas_no, equal_var=False)
print(f"Welch t-test:  t={t_stat:.3f}, p={t_p:.4f}")

print(f"\nBioMedLM reference: yes=0.3217, no=0.3030, MWU p=0.0025")

In [ ]:
# Cell 5 — Per-layer label comparison

print(f"\nPer-layer EAS mean by label (selected layers):")
print(f"{'Layer':<8} {'yes mean':<14} {'no mean':<14} {'diff'}")
print("-" * 50)
for l in [0, 5, 10, 15, 20, 21, 22, 23, 24, 25, 28, 31]:
    y = eas[yes_mask, l].mean()
    n = eas[no_mask,  l].mean()
    print(f"{l:<8} {y:<14.4f} {n:<14.4f} {y-n:+.4f}")

In [ ]:
# Cell 6 — Per-layer Welch t-tests with FDR correction

from scipy.stats import false_discovery_control

layer_t_stats = []
layer_p_vals  = []

for l in range(n_layers):
    t, p = stats.ttest_ind(eas[yes_mask, l], eas[no_mask, l], equal_var=False)
    layer_t_stats.append(t)
    layer_p_vals.append(p)

layer_p_vals  = np.array(layer_p_vals)
layer_t_stats = np.array(layer_t_stats)

# BH FDR correction
rejected = false_discovery_control(layer_p_vals, method="bh") < 0.05

print("Per-layer Welch t-test with BH FDR correction:")
print(f"Layers surviving FDR (p_adj < 0.05): {np.where(rejected)[0].tolist()}")
print(f"Min raw p-value: layer {layer_p_vals.argmin()}, p={layer_p_vals.min():.4f}")
print(f"BioMedLM: no layers survived FDR correction")

In [ ]:
# Cell 7 — Late-layer slope analysis (layers 24-31)

late_layers = np.arange(24, 32)

yes_slopes, no_slopes = [], []

for i in range(len(eas)):
    traj = eas[i, 24:]
    slope, _, _, _, _ = stats.linregress(late_layers, traj)
    if labels[i] == "yes":
        yes_slopes.append(slope)
    else:
        no_slopes.append(slope)

yes_slopes = np.array(yes_slopes)
no_slopes  = np.array(no_slopes)

t_slope, p_slope = stats.ttest_ind(yes_slopes, no_slopes, equal_var=False)

print(f"Late-layer slope analysis (layers 24-31):")
print(f"  yes mean slope: {yes_slopes.mean():+.4f}")
print(f"  no  mean slope: {no_slopes.mean():+.4f}")
print(f"  Welch t={t_slope:.3f}, p={p_slope:.4f}")
print(f"\nBioMedLM: yes=+0.0645, no=+0.0641, p=0.9161 (no difference)")

In [ ]:
# Cell 8 — Summary and save

summary = {
    "model_id": "BioMistral/BioMistral-7B",
    "n_samples": int(len(common_pubids)),
    "n_layers":  int(n_layers),
    "eas_components": ["T2_prob_mass", "T4_kl_rag_vs_sup"],
    "t1_excluded": True,
    "final_layer_eas": {
        "yes": {
            "mean": float(final_eas_yes.mean()),
            "std":  float(final_eas_yes.std()),
            "n":    int(yes_mask.sum())
        },
        "no": {
            "mean": float(final_eas_no.mean()),
            "std":  float(final_eas_no.std()),
            "n":    int(no_mask.sum())
        },
        "mwu_stat":    float(mwu_stat),
        "mwu_p":       float(mwu_p),
        "welch_t":     float(t_stat),
        "welch_p":     float(t_p),
        "significant": bool(mwu_p < 0.05)
    },
    "fdr_surviving_layers": np.where(rejected)[0].tolist(),
    "late_layer_slope": {
        "yes_mean": float(yes_slopes.mean()),
        "no_mean":  float(no_slopes.mean()),
        "p_value":  float(p_slope)
    },
    "biomedlm_reference": {
        "final_eas_yes": 0.3217,
        "final_eas_no":  0.3030,
        "mwu_p":         0.0025
    },
    "per_layer_summary": [
        {
            "layer":    int(l),
            "yes_mean": float(eas[yes_mask, l].mean()),
            "no_mean":  float(eas[no_mask,  l].mean()),
            "t_stat":   float(layer_t_stats[l]),
            "p_raw":    float(layer_p_vals[l]),
            "fdr_sig":  bool(rejected[l])
        }
        for l in range(n_layers)
    ],
    "sample_eas": [
        {
            "pubid":         int(common_pubids[i]),
            "label":         str(labels[i]),
            "eas_trajectory": eas[i].tolist(),
            "final_eas":     float(eas[i, -1])
        }
        for i in range(len(common_pubids))
    ]
}

output_path = os.path.join(OUTPUT_DIR, "biomistral_eas_results.json")
with open(output_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 55)
print("BIOMISTRAL EAS ASSEMBLY — COMPLETE")
print("=" * 55)
print(f"Final-layer EAS: yes={final_eas_yes.mean():.4f}, no={final_eas_no.mean():.4f}")
print(f"MWU p={mwu_p:.4f} | Significant: {mwu_p < 0.05}")
print(f"FDR surviving layers: {np.where(rejected)[0].tolist()}")
print(f"\nBioMedLM reference:")
print(f"  yes=0.3217, no=0.3030, MWU p=0.0025")
print(f"\nSaved to {output_path}")
print("10_biomistral_eas_assembly: complete")